# **GENERAL DATA PREP**

*One-time text cleaning applied to the entire dataset (e.g., lowercasing, removing noise/HTML tags, universal stop word removal, basic tokenization). Finalizing the train/test/validation split.*

In [ ]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

## Libraries

In [ ]:
import sys
import os
import pandas as pd

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from utils import *
from general_preprocessing import *

## Data

In [ ]:
dataset_original = load_dataset('../data/atlanta_restaurant_slice_2023.csv')

In [ ]:
dataset_original

,title,categoryName,website,url,reviewsCount,stars,text
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...
...,...,...,...,...,...,...,...
53561,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Friday night dinner was Chicken Francaise. Jor...
53562,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Great dinner.... Yay Jordan!!!!!!
53563,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was our server and he was fantastic! Gr...
53564,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was an amazing server! Great delicious...


In [ ]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53566 entries, 0 to 53565
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         53566 non-null  object 
 1   categoryName  53566 non-null  object 
 2   website       50600 non-null  object 
 3   url           53566 non-null  object 
 4   reviewsCount  53566 non-null  int64  
 5   stars         53566 non-null  float64
 6   text          53566 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 2.9+ MB


| 🏷️ **Column Name** | 📝 **Description** |
|:-------------------|:-------------------|
|**title** | Name of the restaurant |
|**categoryName** | Labels that describe the restaurant's cuisine type |
|**website** | URL of the restaurant's webpage |
|**url** | URL of the restaurant's Google Maps page |
|**reviewsCount** | Total number of reviews for the restaurant at the time of scraping |
|**stars** | Customer rating (1 to 5) |
|**text** | Text of the review |

In [ ]:
dataset = dataset_original.copy()

In [ ]:
dataset[dataset['title'] == 'Johnny Rockets']['reviewsCount'].unique()

array([   5, 1126])

In [ ]:
# TODO: Preprocess reviewsCount inconsistencies, keep always the higher value in case the same restaurant has different values

dataset = dataset_original.copy()

# Replace each restaurant's reviewsCount with the max reviewsCount for that restaurant
dataset['reviewsCount'] = dataset.groupby('title')['reviewsCount'].transform('max')



In [ ]:
dataset[dataset['title'] == 'Johnny Rockets']['reviewsCount'].unique()

array([1126])

## 2. Preprocessing

In [ ]:
dataset['cleaned_review_body'] = dataset['text'].apply(
    lambda x: main_pipeline(
        raw_text=x,
        no_stopwords=True,
        custom_stopwords=['example1', 'example2'],
        convert_diacritics=True,
        lowercase=True,
        lemmatized=True,
        list_pos=['n', 'v', 'a', 'r', 's'],
        stemmed=False,
        pos_tags_list='no_pos',
        tokenized_output=False,
        no_emojis=True,
        no_hashtags=True,
        hashtag_retain_words=True,
        no_newlines=True,
        no_urls=True,
        no_punctuation=True
    )
)

In [ ]:
dataset[['text', 'cleaned_review_body']].iloc[53].values

array(['Excellent food and vibe - our server seemed to be in a rush though. The oysters and the branzino were excellent!',
       'excellent food vibe - server seem rush though oyster branzino excellent'],
      dtype=object)

In [ ]:
preproc_sentences = [nltk.word_tokenize(text) for text in dataset['cleaned_review_body']]

In [ ]:
preproc_sentences

[['one',
  'word',
  'amaze',
  'red',
  'fish',
  'halibut',
  'fry',
  'rice',
  'broccolini',
  'dish',
  'die',
  'chef',
  'brandon',
  'fantastic',
  'definitely',
  'visit',
  'again'],
 ['first', 'time', 'food', 'great', 'waiter', 'excellent'],
 ['recently',
  'pleasure',
  'din',
  'optimist',
  'atlanta',
  'ga',
  'let',
  'tell',
  'absolute',
  'treat',
  'moment',
  'walk',
  'know',
  'fantastic',
  'din',
  'experience',
  'first',
  'restaurant',
  'offer',
  'indoor',
  'outdoor',
  'seat',
  'option',
  'make',
  'versatile',
  'choice',
  'occasion',
  'whether',
  'prefer',
  'cozy',
  'ambiance',
  'indoor',
  'din',
  'area',
  'refresh',
  'outdoor',
  'patio',
  'optimist',
  'get',
  'cover',
  'kick',
  'even',
  'order',
  'signature',
  'margarita',
  'disappoint',
  'cocktail',
  'expertly',
  'craft',
  'perfectly',
  'balance',
  'incredibly',
  'tasty',
  'set',
  'tone',
  'delightful',
  'meal',
  'ahead',
  'one',
  'thing',
  'stand',
  'right',
  '

In [ ]:
cooc_matrix = cooccurrence_matrix_sentence_generator(preproc_sentences, sentence_cooc=False, window_size=1)

Computing co-occurrences: 100%|██████████| 53566/53566 [00:02<00:00, 20078.55it/s]


                   food  good  great  service  order  place  get   go  time  \
food                146  4330   5133     2345    441    375  632  224   291   
good               4330   238    202     1897    116    479  266  108   204   
great              5133   202     94     3448     61   1065   95   66   288   
service            2345  1897   3448       38     61     85   95  131   101   
order               441   116     61       61    146    404  477  285   252   
...                 ...   ...    ...      ...    ...    ...  ...  ...   ...   
fastfriendlyclean     0     0      0        0      0      0    0    0     0   
emcanto               0     0      0        0      0      0    0    0     0   
nices                 0     0      0        0      0      0    0    0     0   
niceeee               0     0      0        0      0      0    0    0     0   
awssome               0     0      0        0      0      0    0    0     0   

                   come  ...  goob  luvd  pizzaaaaa

In [ ]:
display(cooc_matrix)

,food,good,great,service,order,place,get,go,time,come,...,goob,luvd,pizzaaaaaaa,xpenzv,xcelent,fastfriendlyclean,emcanto,nices,niceeee,awssome
food,146,4330,5133,2345,441,375,632,224,291,448,...,0,0,0,0,0,0,0,0,0,0
good,4330,238,202,1897,116,479,266,108,204,71,...,0,0,0,0,0,0,0,0,0,0
great,5133,202,94,3448,61,1065,95,66,288,45,...,0,0,0,0,0,0,0,0,0,0
service,2345,1897,3448,38,61,85,95,131,101,54,...,0,0,0,0,0,0,0,0,0,0
order,441,116,61,61,146,404,477,285,252,137,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fastfriendlyclean,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
emcanto,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
nices,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
niceeee,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
